# CAWOT-CM V0 — Kaggle end-to-end

**V0 plan (clean baseline, no qproxy):**
1. Random sample 50K (image, caption) entries from the train pool
2. Extract CLIP ViT-B/16 embeddings
3. Hold out 2K pairs as val
4. FAISS k-means (K=150, spherical) over the 48K train pool
5. Two coresets at 20% budget: Random vs V0 (farthest-from-centroid)
6. Fine-tune CLIP-B/16 (last 4 layers + InfoNCE) on each coreset
7. Eval image-text retrieval R@1/5/10 on val

**Setup (Kaggle UI):**
- Settings → Accelerator: **GPU P100**
- Settings → Internet: **On**
- Add Data → friend's image dataset: `vnhtbo/pab-eccv26-track4-train-webp-part-01-05`

Annotations (`imgs_N.json`) are downloaded from HuggingFace at runtime in step 4.

## 1. Sanity check env

In [ ]:
!nvidia-smi -L
!ls /kaggle/input/

## 2. Clone repo

In [ ]:
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm

## 3. Install dependencies (~2 min)

In [ ]:
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub

## 4. Get annotation JSONL files

Download `train/imgs_N.json` from HuggingFace (`TruongVox/Cawot-dataset`). ~390 MB total, ~2-3 min on Kaggle.

Alternative: if friend has uploaded annotations as a Kaggle dataset, skip this cell and just point `ANNOTATIONS_DIR` in step 5 at that mount.

In [ ]:
from huggingface_hub import snapshot_download

annotations_dir = snapshot_download(
    repo_id="TruongVox/Cawot-dataset",
    repo_type="dataset",
    allow_patterns="train/imgs_*.json",
    local_dir="/kaggle/working/hf_cache",
)
ANNOTATIONS_DIR = f"{annotations_dir}/train"
!ls {ANNOTATIONS_DIR} | head
print(f"\nANNOTATIONS_DIR = {ANNOTATIONS_DIR}")

## 5. Patch config

Set the 2 paths Kaggle uses. If you added the image dataset under a different slug, change `IMAGE_ROOT`.

In [ ]:
import yaml

IMAGE_ROOT = "/kaggle/input/pab-eccv26-track4-train-webp-part-01-05"
# ANNOTATIONS_DIR was set in the previous cell

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["image_root"] = IMAGE_ROOT
cfg["data"]["annotations_dir"] = ANNOTATIONS_DIR

with open("config.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(yaml.dump(cfg, sort_keys=False))

## 6. Sanity check that one image resolves

Catches path mismatches before the long run.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/cawot-cm")
from src.data import build_pool

anns, shard_roots = build_pool(
    annotations_dir=cfg["data"]["annotations_dir"],
    image_root=cfg["data"]["image_root"],
    sample_size=100,
    seed=cfg["seed"],
)
print(f"loaded {len(anns)} annotations")
print(f"shards on disk: {sorted(shard_roots)[:5]}...")
print(f"first annotation: {anns[0]}")

# Resolve one image
from src.data import TrainPoolDataset
from torchvision import transforms
tx = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor()])
ds = TrainPoolDataset(anns, shard_roots, image_transform=tx)
sample = ds[0]
print(f"\nfirst image shape: {sample['image'].shape}")
print(f"first caption: {sample['caption'][:80]}...")
print("\n✓ images resolve correctly")

## 7. Run V0 end-to-end

On Kaggle P100 with 50K pool, 20% budget:
- Extract embeddings (50K): ~15-20 min
- Cluster (48K, K=150): ~30 sec
- Train each coreset (~9.6K samples × 3 epochs): ~15 min each
- Eval each: ~2 min
- **Total: ~50-70 min**

All intermediate outputs cached under `/kaggle/working/outputs/` — safe to interrupt and re-run.

In [ ]:
!python scripts/run_v0.py --config config.yaml

## 8. Results

In [ ]:
import json, pandas as pd
with open("/kaggle/working/outputs/eval/summary.json") as f:
    summary = json.load(f)
df = pd.DataFrame(summary).T
df

## 9. What to look for

- **`mean_R@1`**: average of t2i and i2t recall at 1 on the 2K held-out val pairs.
- **Expected ordering**: `v0 > random > zeroshot`. Typical V0-over-Random gap: 0.5–1.5%.
- **If V0 ≤ Random**: bug in selection or K is wrong scale for 48K pool. Sanity check before V1.
- **If both barely beat zeroshot**: training collapsed. Check LR (1e-5 default), batch size, augmentation.

To make these checkpoints downloadable from the notebook output, commit (Kaggle 'Save Version' → 'Save & Run All'). The `/kaggle/working/outputs/` folder is persisted as notebook output.